# Custom MT: Push-v3 + Pick-Place-v3 Results

Notebook για το custom multi-task experiment:

```text
push-v3 + pick-place-v3
```

Το shared PPO policy βλέπει:

```text
Meta-World observation + one-hot task ID
push-v3       -> [1, 0]
pick-place-v3 -> [0, 1]
```

## Load results


In [ ]:
from pathlib import Path
import pandas as pd # type: ignore
import numpy as np # type: ignore
import matplotlib.pyplot as plt # type: ignore

EVAL_DIR = Path("push_pickplace_eval_results")

SUMMARY_PATH = EVAL_DIR / "push_pickplace_summary.csv"
BY_SEED_PATH = EVAL_DIR / "push_pickplace_summary_by_seed.csv"
RAW_PATH = EVAL_DIR / "push_pickplace_raw_episodes.csv"
PIVOT_PATH = EVAL_DIR / "push_pickplace_success_rate_pivot.csv"
SKIPPED_PATH = EVAL_DIR / "skipped_models.csv"

summary = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.exists() else None
by_seed = pd.read_csv(BY_SEED_PATH) if BY_SEED_PATH.exists() else None
raw = pd.read_csv(RAW_PATH) if RAW_PATH.exists() else None
pivot = pd.read_csv(PIVOT_PATH) if PIVOT_PATH.exists() else None

print("Summary exists:", summary is not None, SUMMARY_PATH)
print("By-seed exists:", by_seed is not None, BY_SEED_PATH)
print("Raw exists:", raw is not None, RAW_PATH)
print("Pivot exists:", pivot is not None, PIVOT_PATH)

if summary is not None:
    print("Summary shape:", summary.shape)
    display(summary.head())
    print(summary.columns.tolist())
else:
    print("Run evaluation first or update EVAL_DIR.")


## Final summary table

In [ ]:
if summary is not None:
    display(summary.sort_values(["config", "task_name"]))
    success_pivot = summary.pivot_table(index="config", columns="task_name", values="success_rate")
    display(success_pivot)
else:
    print("No summary loaded.")


## Success rate by config/task


In [ ]:
if summary is not None:
    success_pivot = summary.pivot_table(index="config", columns="task_name", values="success_rate")
    configs = list(success_pivot.index)
    tasks = list(success_pivot.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(tasks), 1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, task in enumerate(tasks):
        ax.bar(x + (i - (len(tasks)-1)/2) * width, success_pivot[task], width, label=task)
    ax.set_title("Custom MT Push/Pick-Place: Success Rate by Config")
    ax.set_xlabel("PPO config")
    ax.set_ylabel("Success rate")
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


## Mean success across tasks

Χρήσιμο για να διαλέξεις το καλύτερο config συνολικά.


In [ ]:
if summary is not None:
    config_summary = (
        summary.groupby("config")
        .agg(
            mean_success=("success_rate", "mean"),
            min_success=("success_rate", "min"),
            mean_return=("avg_return", "mean"),
            mean_first_success_step=("avg_first_success_step", "mean"),
            total_episodes=("episodes", "sum"),
        )
        .reset_index()
        .sort_values(["mean_success", "min_success", "mean_return"], ascending=[False, False, False])
    )
    display(config_summary)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(config_summary["config"], config_summary["mean_success"])
    ax.set_title("Custom MT Push/Pick-Place: Mean Success Across Tasks")
    ax.set_xlabel("PPO config")
    ax.set_ylabel("Mean success rate")
    ax.set_ylim(0, 1.05)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


## Average return by config/task

In [ ]:
if summary is not None:
    return_pivot = summary.pivot_table(index="config", columns="task_name", values="avg_return")
    display(return_pivot)

    configs = list(return_pivot.index)
    tasks = list(return_pivot.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(tasks), 1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, task in enumerate(tasks):
        ax.bar(x + (i - (len(tasks)-1)/2) * width, return_pivot[task], width, label=task)
    ax.set_title("Custom MT Push/Pick-Place: Average Return by Config")
    ax.set_xlabel("PPO config")
    ax.set_ylabel("Average return")
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


## First success step

In [ ]:
if summary is not None and "avg_first_success_step" in summary.columns:
    fs_pivot = summary.pivot_table(index="config", columns="task_name", values="avg_first_success_step")
    display(fs_pivot)

    configs = list(fs_pivot.index)
    tasks = list(fs_pivot.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(tasks), 1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, task in enumerate(tasks):
        ax.bar(x + (i - (len(tasks)-1)/2) * width, fs_pivot[task], width, label=task)
    ax.set_title("Custom MT Push/Pick-Place: Average First Success Step")
    ax.set_xlabel("PPO config")
    ax.set_ylabel("Average first success step")
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No first-success-step column found.")


## Seed stability


In [ ]:
if by_seed is not None:
    display(by_seed.sort_values(["config", "task_name", "eval_seed"]))

    for task in sorted(by_seed["task_name"].unique()):
        d_task = by_seed[by_seed["task_name"] == task]
        fig, ax = plt.subplots(figsize=(8, 5))
        for config in sorted(d_task["config"].unique()):
            d = d_task[d_task["config"] == config].sort_values("eval_seed")
            ax.plot(d["eval_seed"], d["success_rate"], marker="o", label=config)
        ax.set_title(f"Success Across Eval Seeds — {task}")
        ax.set_xlabel("Eval seed")
        ax.set_ylabel("Success rate")
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No summary_by_seed CSV found.")


## Raw episode analysis

In [ ]:
if raw is not None:
    display(raw.head())
    print(raw.columns.tolist())

    raw_grouped = (
        raw.groupby(["config", "task_name", "eval_seed"])
        .agg(
            success_rate=("success", "mean"),
            avg_return=("return", "mean"),
            std_return=("return", "std"),
            avg_episode_length=("episode_length", "mean"),
            avg_first_success_step=("first_success_step", "mean"),
            episodes=("success", "count"),
        )
        .reset_index()
    )
    display(raw_grouped)
else:
    print("No raw episodes CSV found.")


## Best config selection

Ranking με βάση `mean_success`, μετά `min_success`, μετά `mean_return`.


In [ ]:
if summary is not None:
    ranking = (
        summary.groupby("config")
        .agg(
            mean_success=("success_rate", "mean"),
            min_success=("success_rate", "min"),
            mean_return=("avg_return", "mean"),
            mean_first_success_step=("avg_first_success_step", "mean"),
        )
        .reset_index()
        .sort_values(["mean_success", "min_success", "mean_return"], ascending=[False, False, False])
    )
    display(ranking)

    best = ranking.iloc[0]
    print("Best config:", best["config"])
    print("Mean success:", round(best["mean_success"], 3))
    print("Minimum task success:", round(best["min_success"], 3))


## Save figures


In [ ]:
fig_dir = Path("push_pickplace_figures")
fig_dir.mkdir(parents=True, exist_ok=True)
saved = []

if summary is not None:
    success_pivot = summary.pivot_table(index="config", columns="task_name", values="success_rate")
    configs = list(success_pivot.index)
    tasks = list(success_pivot.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(tasks), 1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, task in enumerate(tasks):
        ax.bar(x + (i - (len(tasks)-1)/2) * width, success_pivot[task], width, label=task)
    ax.set_title("Custom MT Push/Pick-Place: Success Rate by Config")
    ax.set_xlabel("PPO config")
    ax.set_ylabel("Success rate")
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    out = fig_dir / "push_pickplace_success_by_config.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    saved.append(out)

    ranking = summary.groupby("config").agg(mean_success=("success_rate", "mean")).reset_index()
    ranking = ranking.sort_values("mean_success", ascending=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(ranking["config"], ranking["mean_success"])
    ax.set_title("Custom MT Push/Pick-Place: Mean Success Across Tasks")
    ax.set_xlabel("PPO config")
    ax.set_ylabel("Mean success rate")
    ax.set_ylim(0, 1.05)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    out = fig_dir / "push_pickplace_mean_success.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    saved.append(out)

    return_pivot = summary.pivot_table(index="config", columns="task_name", values="avg_return")

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, task in enumerate(tasks):
        ax.bar(x + (i - (len(tasks)-1)/2) * width, return_pivot[task], width, label=task)
    ax.set_title("Custom MT Push/Pick-Place: Average Return by Config")
    ax.set_xlabel("PPO config")
    ax.set_ylabel("Average return")
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    out = fig_dir / "push_pickplace_return_by_config.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    saved.append(out)

print("Saved figures:")
for p in saved:
    print(" -", p)
